# MCI-GRU Ablation and Evaluation Loop - Google Colab

Run a controlled ablation matrix against `paper_faithful`, collect Phase 4 evaluation artifacts, and produce a decision table you can use to decide which architecture/data/graph flags deserve to become defaults.

The workbook saves all runs and comparison artifacts to Google Drive.

## 1. Mount Drive, Clone Repo, Install Dependencies

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys

drive.mount('/content/drive')

REPO_URL = 'https://github.com/magilliam27/MCI-GRU.git'
BRANCH = 'main'
REPO_DIR = Path('/content/MCI-GRU')
DRIVE_ROOT = Path('/content/drive/MyDrive/MCI-GRU-Ablations')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    os.chdir(REPO_DIR)
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())

!python -m pip install -q --upgrade pip setuptools wheel
!python -m pip install -q -r requirements.txt
!python -m pip install -q -e ".[dev,tracking,fred]"

REQUIRE_GPU = True  # Set False only for a deliberate CPU dry run.

try:
    import torch
except ImportError as exc:
    raise RuntimeError('PyTorch is not installed; rerun this setup cell.') from exc

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    device_idx = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device_idx)
    print('CUDA version:', torch.version.cuda)
    print('GPU:', torch.cuda.get_device_name(device_idx))
    print(f'GPU memory: {props.total_memory / (1024 ** 3):.1f} GiB')
else:
    msg = (
        'No CUDA GPU is visible in the notebook kernel. In Colab, choose '
        'Runtime -> Change runtime type -> GPU, then restart and run from the top.'
    )
    if REQUIRE_GPU:
        raise RuntimeError(msg)
    print('WARNING:', msg)

child_probe = subprocess.run(
    [
        sys.executable,
        '-c',
        "import torch; "
        "print('Child Torch:', torch.__version__); "
        "print('Child CUDA available:', torch.cuda.is_available()); "
        "print('Child CUDA version:', torch.version.cuda); "
        "print('Child GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO_GPU')",
    ],
    text=True,
    capture_output=True,
)
print('Child Python:', sys.executable)
print(child_probe.stdout.strip())
if child_probe.returncode != 0:
    print(child_probe.stderr)
    raise RuntimeError('Child Python CUDA probe failed; rerun setup after fixing PyTorch.')
child_cuda_visible = 'Child CUDA available: True' in child_probe.stdout
if REQUIRE_GPU and not child_cuda_visible:
    msg = (
        'The training subprocess cannot see CUDA, so run_experiment.py would train on CPU. '
        'Restart the Colab GPU runtime and rerun setup; if this persists, reinstall a CUDA-enabled torch wheel.'
    )
    raise RuntimeError(msg)

print(f'Repo: {REPO_DIR}')
print(f'Drive outputs: {DRIVE_ROOT}')

## 2. Global Run Configuration

Edit this cell first. Start with `FAST_MODE=True`; once the loop is healthy, set it to `False` for full ablations.

In [ ]:
from datetime import datetime

# Data file can live in /content/MCI-GRU or /content/drive/MyDrive/MCI-GRU-Data.
DATA_FILE = 'sp500_2019_universe_data.csv'
GDRIVE_DATA_DIR = Path('/content/drive/MyDrive/MCI-GRU-Data')

# Split dates include > label_t calendar-day embargo by default.
TRAIN_START = '2019-01-01'
TRAIN_END = '2023-12-31'
VAL_START = '2024-01-08'
VAL_END = '2024-12-31'
TEST_START = '2025-01-08'
TEST_END = '2025-12-31'

# FAST_MODE is for validating the notebook and ranking obvious failures.
# Full mode should be used for final decisions.
FAST_MODE = True
FAST_NUM_MODELS = 1
FAST_NUM_EPOCHS = 3
FULL_NUM_MODELS = 10
FULL_NUM_EPOCHS = 100

NUM_MODELS = FAST_NUM_MODELS if FAST_MODE else FULL_NUM_MODELS
NUM_EPOCHS = FAST_NUM_EPOCHS if FAST_MODE else FULL_NUM_EPOCHS

BATCH_SIZE = 32
LEARNING_RATE = '5e-5'
SEED = 42
BOOTSTRAP_RESAMPLES = 200 if FAST_MODE else 1000

# Optional feature inputs.
FRED_API_KEY = ''
os.environ['FRED_API_KEY'] = FRED_API_KEY
SECTOR_MAP_CSV = ''  # Example: data/raw/market/sp500_sector_map.csv
PIT_UNIVERSE_CSV = ''  # Example: data/raw/market/sp500_pit_universe.csv

RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
ABLATION_ROOT = DRIVE_ROOT / ('fast' if FAST_MODE else 'full') / RUN_TAG
ABLATION_ROOT.mkdir(parents=True, exist_ok=True)

print('Ablation root:', ABLATION_ROOT)
print('Mode:', 'FAST sanity' if FAST_MODE else 'FULL decision run')
print('Models:', NUM_MODELS, 'Epochs:', NUM_EPOCHS, 'Bootstrap resamples:', BOOTSTRAP_RESAMPLES)

## 3. Data Availability Check

In [ ]:
import pandas as pd

repo_data = REPO_DIR / DATA_FILE
drive_data = GDRIVE_DATA_DIR / DATA_FILE

if not repo_data.exists() and drive_data.exists():
    !cp "{drive_data}" "{repo_data}"

if not repo_data.exists():
    raise FileNotFoundError(f'Missing {DATA_FILE}. Put it in {repo_data} or {drive_data}.')

preview = pd.read_csv(repo_data, usecols=['dt', 'kdcode'])
preview['dt'] = pd.to_datetime(preview['dt'])
print(f'Rows: {len(preview):,}')
print(f'Stocks: {preview.kdcode.nunique():,}')
print(f'Dates: {preview.dt.min().date()} to {preview.dt.max().date()}')
del preview

## 4. Define Ablation Matrix

Each ablation inherits the same data split, seed, run budget, and Phase 4 evaluation settings. Only the listed overrides vary.

In [ ]:
ABLATIONS = [
    {
        'name': 'paper_faithful',
        'description': 'Control group: legacy architecture, scalar edges, MSE, val_loss selection.',
        'overrides': ['+experiment=paper_faithful'],
    },
    {
        'name': 'modern_defaults',
        'description': 'Current base config: AdamW, IC selection, combined loss, trunk regularisation, fused GRU path.',
        'overrides': [],
    },
    {
        'name': 'a1_a2_cross_attention',
        'description': 'Modern defaults plus direct temporal-to-graph cross-stream fusion.',
        'overrides': ['model.use_a1_a2_cross_attention=true'],
    },
    {
        'name': 'transformer_temporal',
        'description': 'Modern defaults with causal transformer temporal encoder.',
        'overrides': ['model.temporal_encoder=transformer'],
    },
    {
        'name': 'topk20_abs_graph',
        'description': 'Top-k graph selection by absolute correlation while preserving signed edge features.',
        'overrides': ['graph.top_k=20', 'graph.top_k_metric=abs_corr'],
    },
    {
        'name': 'lead_lag_snapshot_age',
        'description': 'Graph edges include lead-lag columns and snapshot age.',
        'overrides': ['graph.use_lead_lag_features=true', 'graph.append_snapshot_age_days=true'],
    },
    {
        'name': 'rank_gauss_norm',
        'description': 'Train-fitted rank-Gaussian normalisation instead of z-score.',
        'overrides': ['data.normalisation=rank_gauss'],
    },
    {
        'name': 'per_split_filter',
        'description': 'Per-split completeness filtering as a survivorship-bias mitigation.',
        'overrides': ['data.filter_stocks_per_split=true'],
    },
]

if SECTOR_MAP_CSV:
    ABLATIONS.append({
        'name': 'sector_relation',
        'description': 'Dual GAT branch with static sector relation edges.',
        'overrides': ['graph.use_sector_relation=true', f'graph.sector_map_csv={SECTOR_MAP_CSV}'],
    })

if PIT_UNIVERSE_CSV:
    ABLATIONS.append({
        'name': 'pit_universe',
        'description': 'Point-in-time universe row filtering.',
        'overrides': ['data.use_pit_universe=true', f'data.pit_universe_csv={PIT_UNIVERSE_CSV}'],
    })

pd.DataFrame(ABLATIONS)[['name', 'description', 'overrides']]

## 5. Helpers: Run, Collect, Score

In [ ]:
import json
import subprocess
import sys
from pathlib import Path
import numpy as np

BASE_OVERRIDES = [
    'features=with_momentum',
    'data.source=csv',
    f'data.filename={DATA_FILE}',
    f'data.train_start={TRAIN_START}',
    f'data.train_end={TRAIN_END}',
    f'data.val_start={VAL_START}',
    f'data.val_end={VAL_END}',
    f'data.test_start={TEST_START}',
    f'data.test_end={TEST_END}',
    f'training.num_models={NUM_MODELS}',
    f'training.num_epochs={NUM_EPOCHS}',
    f'training.batch_size={BATCH_SIZE}',
    f'training.learning_rate={LEARNING_RATE}',
    f'evaluation.bootstrap_resamples={BOOTSTRAP_RESAMPLES}',
    'tracking.enabled=true',
    'tracking.log_predictions=false',
    f'seed={SEED}',
]

def run_ablation(ablation):
    run_dir = ABLATION_ROOT / ablation['name']
    run_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        '-u',
        'run_experiment.py',
        *BASE_OVERRIDES,
        *ablation['overrides'],
        f'experiment_name={ablation["name"]}',
        f'hydra.run.dir={run_dir}',
    ]
    print('\n' + '=' * 100)
    print('Running:', ablation['name'])
    print(ablation['description'])
    print('Command:', ' '.join(cmd))
    print('=' * 100)
    result = subprocess.run(cmd, cwd=REPO_DIR, text=True, capture_output=True)
    (run_dir / 'stdout.log').write_text(result.stdout)
    (run_dir / 'stderr.log').write_text(result.stderr)
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
    return {'name': ablation['name'], 'run_dir': str(run_dir), 'returncode': result.returncode}

def flatten_json(prefix, obj, out):
    if isinstance(obj, dict):
        for key, value in obj.items():
            flatten_json(f'{prefix}.{key}' if prefix else key, value, out)
    elif isinstance(obj, list):
        out[prefix] = json.dumps(obj)
    else:
        out[prefix] = obj

def load_json(path):
    path = Path(path)
    if path.exists():
        return json.loads(path.read_text())
    return None

def collect_ablation(ablation, status):
    run_dir = Path(status['run_dir'])
    row = {
        'name': ablation['name'],
        'description': ablation['description'],
        'returncode': status['returncode'],
        'run_dir': str(run_dir),
        'overrides': ' '.join(ablation['overrides']),
    }
    for filename, prefix in [
        ('training_summary.json', 'training'),
        ('evaluation_summary.json', 'evaluation'),
        ('run_metadata.json', 'metadata'),
        ('walkforward_summary.json', 'walkforward'),
    ]:
        data = load_json(run_dir / filename)
        if data is not None:
            flatten_json(prefix, data, row)
    return row

def add_decision_columns(df):
    out = df.copy()
    for col in [
        'evaluation.metrics.avg_ic',
        'evaluation.metrics.avg_spearman_corr',
        'evaluation.metrics.sharpe_top_20_newey_west',
        'evaluation.metrics.return_top_20',
        'training.mean_best_val_ic',
    ]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce')
    score = pd.Series(0.0, index=out.index)
    weights = {
        'evaluation.metrics.avg_ic': 0.35,
        'evaluation.metrics.avg_spearman_corr': 0.25,
        'evaluation.metrics.sharpe_top_20_newey_west': 0.25,
        'evaluation.metrics.return_top_20': 0.15,
    }
    for col, weight in weights.items():
        if col in out.columns:
            vals = out[col].astype(float)
            denom = vals.std(skipna=True)
            if denom and np.isfinite(denom) and denom > 0:
                score += weight * ((vals - vals.mean(skipna=True)) / denom).fillna(0.0)
    out['decision_score'] = score
    out['status'] = np.where(out['returncode'].eq(0), 'OK', 'FAILED')
    return out.sort_values(['status', 'decision_score'], ascending=[True, False])

## 6. Run Ablations

This cell can take minutes in fast mode and many hours in full mode. It continues after failures and saves stdout/stderr for each run.

In [ ]:
manifest_path = ABLATION_ROOT / 'ablation_manifest.json'
manifest_path.write_text(json.dumps({'ablations': ABLATIONS, 'base_overrides': BASE_OVERRIDES}, indent=2))

statuses = []
rows = []

for ablation in ABLATIONS:
    status = run_ablation(ablation)
    statuses.append(status)
    rows.append(collect_ablation(ablation, status))

raw_df = pd.DataFrame(rows)
raw_path = ABLATION_ROOT / 'ablation_results_raw.csv'
raw_df.to_csv(raw_path, index=False)

decision_df = add_decision_columns(raw_df)
decision_path = ABLATION_ROOT / 'ablation_decision_table.csv'
decision_df.to_csv(decision_path, index=False)

html_path = ABLATION_ROOT / 'ablation_decision_table.html'
decision_df.to_html(html_path, index=False)

print('Saved raw results:', raw_path)
print('Saved decision table:', decision_path)
print('Saved HTML table:', html_path)

display_cols = [c for c in [
    'status',
    'name',
    'decision_score',
    'evaluation.metrics.avg_ic',
    'evaluation.metrics.avg_spearman_corr',
    'evaluation.metrics.sharpe_top_20_newey_west',
    'evaluation.metrics.return_top_20',
    'training.mean_best_val_ic',
    'run_dir',
] if c in decision_df.columns]
display(decision_df[display_cols])

## 7. Visualize The Decision Table

In [ ]:
import matplotlib.pyplot as plt

plot_df = decision_df[decision_df['status'].eq('OK')].copy()
metric_cols = [
    'decision_score',
    'evaluation.metrics.avg_ic',
    'evaluation.metrics.avg_spearman_corr',
    'evaluation.metrics.sharpe_top_20_newey_west',
    'evaluation.metrics.return_top_20',
]
metric_cols = [c for c in metric_cols if c in plot_df.columns]

if not plot_df.empty and metric_cols:
    fig, axes = plt.subplots(len(metric_cols), 1, figsize=(14, 4 * len(metric_cols)))
    if len(metric_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, metric_cols):
        ordered = plot_df.sort_values(col, ascending=False)
        ax.bar(ordered['name'], ordered[col])
        ax.set_title(col)
        ax.tick_params(axis='x', rotation=35)
        ax.axhline(0, color='black', linewidth=0.8, alpha=0.4)
    plt.tight_layout()
    plot_path = ABLATION_ROOT / 'ablation_metric_bars.png'
    plt.savefig(plot_path, dpi=180, bbox_inches='tight')
    plt.show()
    print('Saved plot:', plot_path)
else:
    print('No successful runs or numeric metrics available to plot.')

## 8. Optional: Walk-Forward Verification For Finalists

After the single-split ablation loop, put the strongest two or three candidates here and run a more realistic walk-forward check.

In [ ]:
RUN_WALKFORWARD = False
FINALISTS = ['paper_faithful', 'modern_defaults']

if RUN_WALKFORWARD:
    wf_rows = []
    lookup = {a['name']: a for a in ABLATIONS}
    for name in FINALISTS:
        ablation = dict(lookup[name])
        ablation['name'] = f'wf_{name}'
        ablation['overrides'] = [
            *lookup[name]['overrides'],
            'training.walkforward.enabled=true',
            'training.walkforward.window_train_years=4',
            'training.walkforward.window_val_months=6',
            'training.walkforward.test_span_months=3',
            'training.walkforward.step_months=6',
            'training.walkforward.max_windows=3' if FAST_MODE else 'training.walkforward.max_windows=null',
        ]
        status = run_ablation(ablation)
        wf_rows.append(collect_ablation(ablation, status))
    wf_df = pd.DataFrame(wf_rows)
    wf_path = ABLATION_ROOT / 'walkforward_finalists.csv'
    wf_df.to_csv(wf_path, index=False)
    display(wf_df)
    print('Saved:', wf_path)
else:
    print('Set RUN_WALKFORWARD=True when you are ready for finalist verification.')

## 9. Inspect Failed Runs

In [ ]:
failed = [s for s in statuses if s['returncode'] != 0]
if not failed:
    print('No failed runs.')
else:
    for item in failed:
        run_dir = Path(item['run_dir'])
        print('\n' + '=' * 80)
        print(item['name'], 'failed. Directory:', run_dir)
        print('- stderr tail -')
        print((run_dir / 'stderr.log').read_text()[-4000:])

## 10. Finalist Confirmation

Use this after the fast ablation screen identifies promising candidates. It reruns a small finalist set with more training budget and writes results to a separate Drive folder so the broad-screen outputs remain unchanged.

In [ ]:
RUN_FINALIST_CONFIRMATION = False

CONFIRMATION_NUM_MODELS = 3
CONFIRMATION_NUM_EPOCHS = 20
CONFIRMATION_BOOTSTRAP_RESAMPLES = 500

CONFIRMATION_RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
FINALIST_CONFIRMATION_ROOT = DRIVE_ROOT / 'finalist_confirmation' / CONFIRMATION_RUN_TAG

FINALIST_CONFIRMATION_RUNS = [
    {
        'name': 'modern_defaults',
        'description': 'Current base config; finalist baseline from the fast screen.',
        'overrides': [],
    },
    {
        'name': 'topk10_abs_graph',
        'description': 'Top-10 graph selection by absolute correlation.',
        'overrides': ['graph.top_k=10', 'graph.top_k_metric=abs_corr'],
    },
    {
        'name': 'topk20_abs_graph',
        'description': 'Top-20 graph selection by absolute correlation; fast-screen leader.',
        'overrides': ['graph.top_k=20', 'graph.top_k_metric=abs_corr'],
    },
    {
        'name': 'topk30_abs_graph',
        'description': 'Top-30 graph selection by absolute correlation.',
        'overrides': ['graph.top_k=30', 'graph.top_k_metric=abs_corr'],
    },
    {
        'name': 'topk20_pos_graph',
        'description': 'Top-20 graph selection by positive correlation only.',
        'overrides': ['graph.top_k=20', 'graph.top_k_metric=corr'],
    },
]

CONFIRMATION_BASE_OVERRIDES = [
    'features=with_momentum',
    'data.source=csv',
    f'data.filename={DATA_FILE}',
    f'data.train_start={TRAIN_START}',
    f'data.train_end={TRAIN_END}',
    f'data.val_start={VAL_START}',
    f'data.val_end={VAL_END}',
    f'data.test_start={TEST_START}',
    f'data.test_end={TEST_END}',
    f'training.num_models={CONFIRMATION_NUM_MODELS}',
    f'training.num_epochs={CONFIRMATION_NUM_EPOCHS}',
    f'training.batch_size={BATCH_SIZE}',
    f'training.learning_rate={LEARNING_RATE}',
    f'evaluation.bootstrap_resamples={CONFIRMATION_BOOTSTRAP_RESAMPLES}',
    'tracking.enabled=true',
    'tracking.log_predictions=false',
    f'seed={SEED}',
]

def run_finalist_confirmation(ablation):
    run_dir = FINALIST_CONFIRMATION_ROOT / ablation['name']
    run_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        '-u',
        'run_experiment.py',
        *CONFIRMATION_BASE_OVERRIDES,
        *ablation['overrides'],
        f'experiment_name=confirm_{ablation["name"]}',
        f'hydra.run.dir={run_dir}',
    ]
    print('\n' + '=' * 100)
    print('Confirming:', ablation['name'])
    print(ablation['description'])
    print('Command:', ' '.join(cmd))
    print('=' * 100)
    result = subprocess.run(cmd, cwd=REPO_DIR, text=True, capture_output=True)
    (run_dir / 'stdout.log').write_text(result.stdout)
    (run_dir / 'stderr.log').write_text(result.stderr)
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
    return {'name': ablation['name'], 'run_dir': str(run_dir), 'returncode': result.returncode}

if RUN_FINALIST_CONFIRMATION:
    FINALIST_CONFIRMATION_ROOT.mkdir(parents=True, exist_ok=True)
    finalist_manifest = {
        'runs': FINALIST_CONFIRMATION_RUNS,
        'base_overrides': CONFIRMATION_BASE_OVERRIDES,
        'source_fast_ablation_root': str(ABLATION_ROOT),
    }
    (FINALIST_CONFIRMATION_ROOT / 'finalist_confirmation_manifest.json').write_text(
        json.dumps(finalist_manifest, indent=2)
    )

    finalist_statuses = []
    finalist_rows = []
    for ablation in FINALIST_CONFIRMATION_RUNS:
        status = run_finalist_confirmation(ablation)
        finalist_statuses.append(status)
        finalist_rows.append(collect_ablation(ablation, status))

    finalist_raw_df = pd.DataFrame(finalist_rows)
    finalist_raw_path = FINALIST_CONFIRMATION_ROOT / 'finalist_confirmation_raw.csv'
    finalist_raw_df.to_csv(finalist_raw_path, index=False)

    finalist_decision_df = add_decision_columns(finalist_raw_df)
    finalist_decision_path = FINALIST_CONFIRMATION_ROOT / 'finalist_confirmation_decision_table.csv'
    finalist_decision_df.to_csv(finalist_decision_path, index=False)
    finalist_html_path = FINALIST_CONFIRMATION_ROOT / 'finalist_confirmation_decision_table.html'
    finalist_decision_df.to_html(finalist_html_path, index=False)

    print('Saved finalist raw results:', finalist_raw_path)
    print('Saved finalist decision table:', finalist_decision_path)
    print('Saved finalist HTML table:', finalist_html_path)

    finalist_display_cols = [c for c in [
        'status',
        'name',
        'decision_score',
        'evaluation.metrics.avg_ic',
        'evaluation.metrics.avg_spearman_corr',
        'evaluation.metrics.sharpe_top_20_newey_west',
        'evaluation.metrics.return_top_20',
        'training.mean_best_val_ic',
        'run_dir',
    ] if c in finalist_decision_df.columns]
    display(finalist_decision_df[finalist_display_cols])
else:
    print('Set RUN_FINALIST_CONFIRMATION=True to run the finalist confirmation suite.')
    print('Finalist output root will be:', FINALIST_CONFIRMATION_ROOT)

## 11. Final Confirmation: Signal Survival Check

Run this only after the finalist screen. It narrows the question to whether the current graph candidate survives a tougher walk-forward check against the modern baseline. The output table includes explicit continuation flags instead of just another broad ranking.

In [ ]:
RUN_FINAL_CONFIRMATION = False

FINAL_CONFIRMATION_NUM_MODELS = 5
FINAL_CONFIRMATION_NUM_EPOCHS = 50
FINAL_CONFIRMATION_BOOTSTRAP_RESAMPLES = 1000
FINAL_CONFIRMATION_MAX_WINDOWS = 5
INCLUDE_RETURN_CHALLENGER = True

FINAL_CONFIRMATION_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
FINAL_CONFIRMATION_ROOT = DRIVE_ROOT / 'final_confirmation' / FINAL_CONFIRMATION_TAG

FINAL_CONFIRMATION_PRIMARY_RUNS = [
    {
        'name': 'modern_defaults',
        'description': 'Modern default baseline; keep this as the control.',
        'overrides': [],
    },
    {
        'name': 'topk30_abs_graph',
        'description': 'Finalist graph candidate from the confirmation screen.',
        'overrides': ['graph.top_k=30', 'graph.top_k_metric=abs_corr'],
    },
]

FINAL_CONFIRMATION_OPTIONAL_RUNS = [
    {
        'name': 'topk10_abs_graph',
        'description': 'Return-focused challenger; include if you can afford one extra run.',
        'overrides': ['graph.top_k=10', 'graph.top_k_metric=abs_corr'],
    },
]

FINAL_CONFIRMATION_RUNS = [
    *FINAL_CONFIRMATION_PRIMARY_RUNS,
    *(FINAL_CONFIRMATION_OPTIONAL_RUNS if INCLUDE_RETURN_CHALLENGER else []),
]

FINAL_CONFIRMATION_BASE_OVERRIDES = [
    'features=with_momentum',
    'data.source=csv',
    f'data.filename={DATA_FILE}',
    f'data.train_start={TRAIN_START}',
    f'data.train_end={TRAIN_END}',
    f'data.val_start={VAL_START}',
    f'data.val_end={VAL_END}',
    f'data.test_start={TEST_START}',
    f'data.test_end={TEST_END}',
    f'training.num_models={FINAL_CONFIRMATION_NUM_MODELS}',
    f'training.num_epochs={FINAL_CONFIRMATION_NUM_EPOCHS}',
    f'training.batch_size={BATCH_SIZE}',
    f'training.learning_rate={LEARNING_RATE}',
    'training.walkforward.enabled=true',
    'training.walkforward.window_train_years=4',
    'training.walkforward.window_val_months=6',
    'training.walkforward.test_span_months=3',
    'training.walkforward.step_months=6',
    f'training.walkforward.max_windows={FINAL_CONFIRMATION_MAX_WINDOWS}',
    f'evaluation.bootstrap_resamples={FINAL_CONFIRMATION_BOOTSTRAP_RESAMPLES}',
    'tracking.enabled=true',
    'tracking.log_predictions=false',
    f'seed={SEED}',
]

def run_final_confirmation(ablation):
    run_dir = FINAL_CONFIRMATION_ROOT / ablation['name']
    run_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        '-u',
        'run_experiment.py',
        *FINAL_CONFIRMATION_BASE_OVERRIDES,
        *ablation['overrides'],
        f'experiment_name=final_{ablation["name"]}',
        f'hydra.run.dir={run_dir}',
    ]
    print('\n' + '=' * 100)
    print('Final confirming:', ablation['name'])
    print(ablation['description'])
    print('Command:', ' '.join(cmd))
    print('=' * 100)
    result = subprocess.run(cmd, cwd=REPO_DIR, text=True, capture_output=True)
    (run_dir / 'stdout.log').write_text(result.stdout)
    (run_dir / 'stderr.log').write_text(result.stderr)
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
    return {'name': ablation['name'], 'run_dir': str(run_dir), 'returncode': result.returncode}

def add_final_confirmation_columns(df):
    out = df.copy()
    metric_cols = [
        'walkforward.mean_best_val_ic_across_windows',
        'walkforward.evaluation.mean_avg_ic_across_windows',
        'walkforward.evaluation.mean_avg_ic_ci_lower_across_windows',
        'walkforward.evaluation.mean_avg_spearman_corr_across_windows',
        'walkforward.evaluation.mean_return_top_20_across_windows',
        'walkforward.evaluation.mean_sharpe_top_20_newey_west_across_windows',
        'walkforward.evaluation.mean_top_20_return_ci_lower_across_windows',
    ]
    for col in metric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce')
    out['status'] = np.where(out['returncode'].eq(0), 'OK', 'FAILED')
    out['ic_ci_pass'] = out.get(
        'walkforward.evaluation.mean_avg_ic_ci_lower_across_windows', pd.Series(np.nan, index=out.index)
    ).gt(0)
    out['top20_ci_pass'] = out.get(
        'walkforward.evaluation.mean_top_20_return_ci_lower_across_windows', pd.Series(np.nan, index=out.index)
    ).gt(0)
    out['both_ci_pass'] = out['ic_ci_pass'] & out['top20_ci_pass']

    score = pd.Series(0.0, index=out.index)
    weights = {
        'walkforward.evaluation.mean_avg_ic_across_windows': 0.35,
        'walkforward.evaluation.mean_avg_spearman_corr_across_windows': 0.25,
        'walkforward.evaluation.mean_sharpe_top_20_newey_west_across_windows': 0.25,
        'walkforward.evaluation.mean_return_top_20_across_windows': 0.15,
    }
    for col, weight in weights.items():
        if col in out.columns:
            vals = out[col].astype(float)
            denom = vals.std(skipna=True)
            if denom and np.isfinite(denom) and denom > 0:
                score += weight * ((vals - vals.mean(skipna=True)) / denom).fillna(0.0)
    out['final_confirmation_score'] = score
    out['continuation_recommendation'] = np.select(
        [out['status'].ne('OK'), out['both_ci_pass'], out['ic_ci_pass'] | out['top20_ci_pass']],
        ['FAILED', 'CONTINUE', 'BORDERLINE'],
        default='STOP',
    )
    return out.sort_values(['status', 'both_ci_pass', 'final_confirmation_score'], ascending=[True, False, False])

if RUN_FINAL_CONFIRMATION:
    FINAL_CONFIRMATION_ROOT.mkdir(parents=True, exist_ok=True)
    final_manifest = {
        'runs': FINAL_CONFIRMATION_RUNS,
        'base_overrides': FINAL_CONFIRMATION_BASE_OVERRIDES,
        'source_finalist_confirmation_root': str(FINALIST_CONFIRMATION_ROOT),
    }
    (FINAL_CONFIRMATION_ROOT / 'final_confirmation_manifest.json').write_text(
        json.dumps(final_manifest, indent=2)
    )

    final_rows = []
    for ablation in FINAL_CONFIRMATION_RUNS:
        status = run_final_confirmation(ablation)
        final_rows.append(collect_ablation(ablation, status))

    final_raw_df = pd.DataFrame(final_rows)
    final_raw_path = FINAL_CONFIRMATION_ROOT / 'final_confirmation_raw.csv'
    final_raw_df.to_csv(final_raw_path, index=False)

    final_decision_df = add_final_confirmation_columns(final_raw_df)
    final_decision_path = FINAL_CONFIRMATION_ROOT / 'final_confirmation_decision_table.csv'
    final_decision_df.to_csv(final_decision_path, index=False)
    final_html_path = FINAL_CONFIRMATION_ROOT / 'final_confirmation_decision_table.html'
    final_decision_df.to_html(final_html_path, index=False)

    print('Saved final raw results:', final_raw_path)
    print('Saved final decision table:', final_decision_path)
    print('Saved final HTML table:', final_html_path)

    final_display_cols = [c for c in [
        'status',
        'name',
        'continuation_recommendation',
        'both_ci_pass',
        'ic_ci_pass',
        'top20_ci_pass',
        'final_confirmation_score',
        'walkforward.n_windows',
        'walkforward.evaluation.mean_avg_ic_across_windows',
        'walkforward.evaluation.mean_avg_ic_ci_lower_across_windows',
        'walkforward.evaluation.mean_avg_spearman_corr_across_windows',
        'walkforward.evaluation.mean_return_top_20_across_windows',
        'walkforward.evaluation.mean_top_20_return_ci_lower_across_windows',
        'walkforward.evaluation.mean_sharpe_top_20_newey_west_across_windows',
        'run_dir',
    ] if c in final_decision_df.columns]
    display(final_decision_df[final_display_cols])
else:
    print('Set RUN_FINAL_CONFIRMATION=True to run the final signal survival check.')
    print('Final confirmation output root will be:', FINAL_CONFIRMATION_ROOT)
    print('Primary runs:', [run['name'] for run in FINAL_CONFIRMATION_PRIMARY_RUNS])
    print('Optional challenger enabled:', INCLUDE_RETURN_CHALLENGER)